# 1 · Setup - die model, corner check and preview

This notebook rebuilds your die in **Blender**, checks it against your corner photos and renders a small **preview**
with timing. The full render is notebook **2_render**. Everything is stored in your Google Drive in the project folder.

**Once before:** in the menu **Runtime → Change runtime type → L4 GPU** (or T4) and save.
The Colab package from the trainer (ZIP file, not unpacked) is in *My Drive → __DRIVE__ → packages*.

**How:** start the cells in order (▶ left of each cell, or *Runtime → Run all*).
At cell 7 you have to compare and confirm yourself - nothing is rendered before that.

## 1 · Settings
The trainer has filled in Drive folder and project. Normally nothing needs to be changed here.
- `IMAGES`: how many renders notebook 2 makes in total - here only used for the time estimate after the preview (cell 8).
- `SAMPLES`: quality (more = nicer, but slower). 32 is a good start.
- `SHARE_…`: shares of the special throw types. The preview uses the same values as the full render.

In [ ]:
DRIVE_FOLDER = "__DRIVE__"        # folder in "My Drive"
PROJECT = "__PROJECT__"           # project folder name (under projects/)
NB_NAME, NB_VERSION = "1_setup", 1   # version of this notebook (to detect updates)
BLENDER_SERIES = "4.5"            # Blender LTS series; the exact version is recorded the first time
IMAGES = 28000                    # renders in total
IMAGES_PER_CHUNK = 250            # rendered in one go, then saved to Drive
SAMPLES = 32                      # quality (more = nicer, but slower)
SEED = 1
SHARE_CLOSE = 0.35                # throws with dice lying close together
SHARE_LEAN_WALL = 0.2             # images with a die leaning against the wall (“?”)
SHARE_KEEP = 0.2                  # “keep and re-roll”: 2-4 dice pushed together at the wall, the rest thrown
SHARE_LEAN_DIE = 0.05             # images with a die leaning against another die (“?”)

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
P = f"{DRIVE}/projects/{PROJECT}"
SCRIPTS = f"{DRIVE}/scripts"
SCENE = f"{P}/scene"
RENDERS = f"{P}/renders"
PREVIEW = f"{P}/renders_preview"
EXPORT = f"{P}/export"
DET = f"{P}/detector"
CLS = f"{P}/classifier"
ESP_DETECTION = "/content/esp-detection"
print("Project folder:", P)

## 2 · Check the GPU
Expected: **NVIDIA L4** (faster) or **Tesla T4**. If it says "no GPU": *Runtime → Change runtime type → L4 GPU*
(or T4), then start again from the top.

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("✔ GPU:", gpu) if gpu else print("✘ no GPU! Runtime → Change runtime type → L4 or T4 GPU, then run everything again.")

## 3 · Connect Google Drive and unpack the newest package
Colab asks whether the notebook may access your Drive → **Allow**.

Then the **newest Colab package** in `packages/` is unpacked (the ZIP file from the trainer - upload it to
*My Drive → __DRIVE__ → packages*, do **not** unpack it). Scripts, notebooks, `project.json`, photos, test images and the
camera setup are updated. **Renders and training results are not touched.**
A package is unpacked only once; a newer package replaces photos, test images and camera setup completely.

In [ ]:
import glob, json, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
PACKAGES = f"{DRIVE}/packages"
os.makedirs(PACKAGES, exist_ok=True)
packages = sorted(glob.glob(f"{PACKAGES}/Colab_{PROJECT}_*.zip"))
state_file = f"{DRIVE}/package_state.json"
state = json.load(open(state_file)) if os.path.exists(state_file) else {}
if not packages:
    print(f"⚠ No package “Colab_{PROJECT}_….zip” in {PACKAGES} - create one in the trainer and upload the ZIP file there.")
elif state.get(PROJECT) == os.path.basename(packages[-1]):
    print("✔ newest package is unpacked already:", os.path.basename(packages[-1]))
else:
    new = packages[-1]
    print("unpacking:", os.path.basename(new))
    with zipfile.ZipFile(new) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        prefix = names[0].split("/")[0]
        # mirror of the laptop: these folders are replaced completely (deleted photos disappear here too)
        for part in ("scripts", "notebooks", f"projects/{PROJECT}/photos", f"projects/{PROJECT}/test_images",
                     f"projects/{PROJECT}/camera_setup"):
            if any(n.startswith(f"{prefix}/{part}/") for n in names) and os.path.isdir(f"{DRIVE}/{part}"):
                shutil.rmtree(f"{DRIVE}/{part}")
        for n in names:
            target = os.path.join(DRIVE, n.split("/", 1)[1])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(n) as src, open(target, "wb") as f:
                shutil.copyfileobj(src, f)
    state[PROJECT] = os.path.basename(new)
    json.dump(state, open(state_file, "w"), indent=1)
    print(f"✔ unpacked ({len(names)} files)")
    try:
        v = json.load(open(f"{DRIVE}/notebooks/version.json")).get(NB_NAME, 0)
        if v > NB_VERSION:
            print(f"⚠ The package contains a NEWER version of this notebook: please open notebooks/{NB_NAME}.ipynb in Drive again.")
    except (OSError, ValueError):
        pass
missing = [p for p in (f"{P}/project.json", f"{SCRIPTS}/render.py") if not os.path.exists(p)]
if missing:
    print("✘ not found:", *missing, sep="\n   ")
else:
    print("✔ project and scripts found")

## 4 · Install Blender (automatic, about 1-3 minutes)
Downloads Blender 4.5 LTS from **download.blender.org** and unpacks it on the Colab disk. The first time the download
is also stored in your Drive (`blender/`) - after that it is faster and it always stays **the same version**.
Colab forgets everything installed in every new session, so this cell runs every time.
At the end it shows whether Blender sees the GPU (expected: **OPTIX** or **CUDA**).

In [ ]:
import json, os, re, shutil, subprocess, urllib.request
CACHE = f"{DRIVE}/blender"
os.makedirs(CACHE, exist_ok=True)
vfile = f"{CACHE}/version.json"
if os.path.exists(vfile):
    version = json.load(open(vfile))["version"]
else:
    html = urllib.request.urlopen(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/").read().decode()
    found = set(re.findall(rf"blender-({re.escape(BLENDER_SERIES)}\.\d+)-linux-x64\.tar\.xz", html))
    version = sorted(found, key=lambda v: int(v.split(".")[-1]))[-1]
    json.dump({"version": version}, open(vfile, "w"))
archive = f"blender-{version}-linux-x64.tar.xz"
if not os.path.exists(f"{CACHE}/{archive}"):
    print("downloading", archive, "…")
    urllib.request.urlretrieve(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/{archive}", f"/content/{archive}")
    shutil.copy(f"/content/{archive}", f"{CACHE}/{archive}")
BLENDER_DIR = f"/content/blender-{version}-linux-x64"
if not os.path.isdir(BLENDER_DIR):
    source = f"/content/{archive}" if os.path.exists(f"/content/{archive}") else f"{CACHE}/{archive}"
    subprocess.run(["tar", "-xf", source, "-C", "/content"], check=True)
subprocess.run("apt-get -qq update > /dev/null; apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
               "libxxf86vm1 libxfixes3 libgl1 libegl1 > /dev/null", shell=True)
BLENDER = f"{BLENDER_DIR}/blender"
print(subprocess.run([BLENDER, "-b", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
test = ("import bpy; p=bpy.context.preferences.addons['cycles'].preferences\n"
        "for t in ('OPTIX','CUDA'):\n"
        "    try:\n        p.compute_device_type=t; p.get_devices()\n"
        "        g=[d.name for d in p.devices if d.type==t]\n"
        "        if g: print('GPU RENDERING:', t, g); break\n"
        "    except TypeError: pass\n"
        "else: print('GPU RENDERING: none - rendering on the CPU (very slow)')")
out = subprocess.run([BLENDER, "-b", "--python-expr", test], capture_output=True, text=True).stdout
print([z for z in out.splitlines() if z.startswith("GPU RENDERING")] or out[-500:])

## 5 · Prepare the photos for Blender
Crops the 6 face photos exactly, straightens them, matches the colours to the white paper and combines them according to
your die net into a **texture atlas**. Also: felt, tray wall, surroundings photos and all sizes/ranges
(`scene/scene.json`). Takes less than a minute.

**Good if:** below you see 6 clean die faces - no paper edge, no shadow, not skewed.
If a face looks bad: in the trainer, draw the yellow frame of this photo again (or take a new photo), build a new Colab
package, upload it and repeat from here.

In [ ]:
import os, subprocess
from IPython.display import Image, display
r = subprocess.run(["python", f"{SCRIPTS}/prepare_textures.py", "--project", P, "--output", SCENE],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
if os.path.exists(f"{SCENE}/atlas.png"):
    display(Image(f"{SCENE}/atlas.png", width=600))

## 6 · Render the 3D die from 8 corners
Renders your 3D die from the same 8 corner views as the 3D preview in the trainer and puts your corner photos below.
Takes about 1 minute.

In [ ]:
r = subprocess.run([BLENDER, "-b", "-P", f"{SCRIPTS}/check_corners.py", "--", "--scene", SCENE],
                   capture_output=True, text=True)
print("\n".join(z for z in r.stdout.splitlines() if "rendered" in z or "✔" in z or "⚠" in z or "Error" in z))
if r.returncode != 0 or "✔ check done" not in r.stdout:
    print("✖ Rendering failed:\n", r.stderr[-2000:])
else:
    r = subprocess.run(["python", f"{SCRIPTS}/contact_sheet.py", "check", "--scene", SCENE, "--project", P,
                        "--output", f"{SCENE}/check.jpg"], capture_output=True, text=True)
    print(r.stdout, r.stderr[-1500:])
    if os.path.exists(f"{SCENE}/check.jpg"):
        display(Image(f"{SCENE}/check.jpg", width=1000))
    else:
        print("✖ The check image was not created (see the output above).")

## 7 · Compare and confirm (required!)
Find the matching rendered view for each of your corner photos (bottom row). Check:
1. **The same symbols** on the three visible faces?
2. **The same rotation** of every symbol (where does the head of a figure point, how do the rows of pips run)?

**If everything matches:** set `CHECK_OK = True` below and run the cell.
**If something is wrong:** correct the die net in the trainer, build a new Colab package, upload it and repeat from cell 5.
Otherwise the model learns a die that does not exist.

The confirmation is stored on Drive and is valid only for exactly this texture atlas - if the die changes, confirm again.

In [ ]:
import hashlib, json, os

def atlas_hash():
    return hashlib.sha1(open(f"{SCENE}/atlas.png", "rb").read() + open(f"{SCENE}/scene.json", "rb").read()).hexdigest()

def check_confirmed():
    f = f"{SCENE}/check_ok.json"
    return (os.path.exists(f) and os.path.exists(f"{SCENE}/atlas.png")
            and json.load(open(f)).get("hash") == atlas_hash())

import time
CHECK_OK = False   # <- set to True after comparing

if CHECK_OK:
    json.dump({"hash": atlas_hash(), "confirmed": time.strftime("%Y-%m-%d %H:%M:%S")}, open(f"{SCENE}/check_ok.json", "w"))
print("✔ Check confirmed - rendering is allowed." if check_confirmed()
      else "✘ Not confirmed yet. Compare first, then set CHECK_OK = True and run this cell.")

## 8 · Preview with timing (about 2-5 minutes)
Renders **30 images** into a separate folder (`renders_preview`) with the same throw types as the full render and
measures the time per image. From that it calculates how long `IMAGES` take. Measured, not estimated.
Measure again (e.g. with other SAMPLES)? Set `PREVIEW_AGAIN = True`.

In [ ]:
import shutil
PREVIEW_AGAIN = False
assert check_confirmed(), "First cell 7: confirm the check."
if PREVIEW_AGAIN and os.path.isdir(PREVIEW):
    shutil.rmtree(PREVIEW)
proc = subprocess.Popen([BLENDER, "-b", "-P", f"{SCRIPTS}/render.py", "--", "--scene", SCENE, "--output", PREVIEW,
                         "--chunk-from", "0", "--chunk-to", "0", "--images", "30", "--seed", "999", "--samples", str(SAMPLES), "--share-close", str(SHARE_CLOSE), "--share-lean-wall", str(SHARE_LEAN_WALL), "--share-keep", str(SHARE_KEEP), "--share-lean-die", str(SHARE_LEAN_DIE)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for z in proc.stdout:
    if "Chunk" in z or "✔" in z or "ERROR" in z or "Error" in z or "render device" in z:
        print(z, end="")
proc.wait()
info = json.load(open(f"{PREVIEW}/chunk_000/done.json"))
s = info["s_per_image"]
print(f"\n⏱ measured: {s:.2f} s per image ({info['device']}, {info['samples']} samples)")
h = IMAGES * s / 3600
print(f"   estimate for {IMAGES} images: {h:.1f} hours ({h * 60:.0f} minutes)")

## 9 · Contact sheet of the preview
Green = die with symbol, **orange = “?”** (tilted/covered, the actual top symbol in brackets).
**Good if:** the images look like real throws, the labels match the dice, and distances/angles roughly match your
camera position. If something looks unreal (too bright, wrong colours …), fix it before the full render runs.

In [ ]:
r = subprocess.run(["python", f"{SCRIPTS}/contact_sheet.py", "renders", "--folder", PREVIEW, "--output",
                    f"{PREVIEW}/sheet.jpg", "--count", "16"], capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
display(Image(f"{PREVIEW}/sheet.jpg", width=1100))

## What next?
1. `notebooks/2_render.ipynb` renders the full set (hours - see the estimate in cell 8),
2. then `notebooks/3_detector.ipynb` (finds the dice in the image)
3. and `notebooks/4_classifier.ipynb` (recognises the top symbol).

**Or all in one go** (e.g. overnight): `notebooks/5_all_in_one.ipynb` renders, trains detector and classifier,
exports both and then ends the session.

All of them are already in your Drive (they came with the package). Real throws labelled in the trainer are used
there as test images - the more, the more honest the measurement.

## End the session (optional)
A connected Colab session uses **compute units even when it does nothing**. If you leave while something is still
running: set `END_SESSION = True` below and start this cell with ▶ - it queues up and ends the session when all cells
before it are done. All results are on Drive, nothing is lost.
**Note:** if a cell before it stops with an error, Colab discards the waiting cells - this one too. The session then
keeps running until Colab disconnects it (or you choose *Runtime → Disconnect and delete runtime*).
With *Run all* nothing happens here as long as `END_SESSION = False`.

In [ ]:
END_SESSION = False   # <- set to True and press ▶ if the session should end after the running cells
if END_SESSION:
    from google.colab import runtime
    print("Ending the session - results are on Drive.")
    runtime.unassign()
else:
    print("Session keeps running (END_SESSION = False).")